Import libraries and path

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import joblib

PROJECT_ROOT = Path.cwd().parent  # notebooks/ -> repo root
INTERIM_DIR = PROJECT_ROOT / "data" / "interim"
MODELS_DIR = PROJECT_ROOT / "models"

MODEL_PATH = MODELS_DIR / "cease_risk_model.joblib"
META_PATH = MODELS_DIR / "metadata.json"

MODEL_PATH.exists(), META_PATH.exists()

(True, True)

Loading Model and metadata

In [2]:
model = joblib.load(MODEL_PATH)

with open(META_PATH, "r") as f:
    meta = json.load(f)

meta.keys(), meta.get("model_name"), len(meta.get("features", []))

(dict_keys(['trained_at', 'model_name', 'lookback_days', 'horizon_days', 'val_roc_auc', 'val_pr_auc', 'features', 'categorical_features', 'numeric_features']),
 'HistGradientBoostingClassifier',
 25)

Loading Validation and test sets

In [3]:
val_path = INTERIM_DIR / "val.parquet"
test_path = INTERIM_DIR / "test.parquet"

val_df = pd.read_parquet(val_path)
test_df = pd.read_parquet(test_path)

val_df.shape, test_df.shape

((296982, 24), (261737, 24))

Prepare features

In [5]:
import numpy as np

# these should match what you used in training
def add_log_features(df):
    df = df.copy()
    for col in ["sum_download_30d", "sum_upload_30d", "avg_talk_time_30d", "avg_hold_time_30d"]:
        if col in df.columns:
            df[f"log1p_{col}"] = np.log1p(df[col].clip(lower=0))
    return df

val_df = add_log_features(val_df)
test_df = add_log_features(test_df)

feature_cols = meta["features"]

missing_val = sorted(set(feature_cols) - set(val_df.columns))
missing_test = sorted(set(feature_cols) - set(test_df.columns))

print("Missing in val:", missing_val)
print("Missing in test:", missing_test)

assert len(missing_val) == 0, "Validation missing required feature columns"
assert len(missing_test) == 0, "Test missing required feature columns"

Missing in val: []
Missing in test: []


Add log features so features match metadata

In [8]:
def add_log_features(df):
    df = df.copy()
    for col in ["sum_download_30d", "sum_upload_30d", "avg_talk_time_30d", "avg_hold_time_30d"]:
        if col in df.columns:
            df[f"log1p_{col}"] = np.log1p(df[col].clip(lower=0))
    return df

val_df = add_log_features(val_df)
test_df = add_log_features(test_df)

Score val/test

In [9]:
val_proba = model.predict_proba(val_df[feature_cols])[:, 1]
test_proba = model.predict_proba(test_df[feature_cols])[:, 1]

val_scored = val_df[[ID_COL, DATE_COL, TARGET]].copy()
val_scored["risk_score"] = val_proba

test_scored = test_df[[ID_COL, DATE_COL, TARGET]].copy()
test_scored["risk_score"] = test_proba

val_scored.head()

,unique_customer_identifier,snapshot_date,target_30d,risk_score
0,2147b7e439b1e4dd597e88ee7cf15a4fc23d27046e7e17...,2024-05-01,0,0.083137
1,214fce5d267b45ba497710c529863fc1c9e733c3255beb...,2024-06-01,1,0.844723
2,21562ee19c61b797d76598c8be6fa2a492ceb840f52938...,2024-05-01,0,0.073508
3,21581257132d92a65c95b626d57dc452e717f6ed94e49f...,2024-06-01,0,0.273790
4,215b291bcfeb532e61e9444947bc23f55350f644f29efe...,2024-05-01,0,0.184566


Compute risk band thresholds from validation

High = top 5%, Medium = next 15% (top 20%), Low = rest.

In [10]:
HIGH_PCT = 0.05
MEDIUM_TOP_PCT = 0.20

q_high = float(np.quantile(val_scored["risk_score"], 1 - HIGH_PCT))
q_medium = float(np.quantile(val_scored["risk_score"], 1 - MEDIUM_TOP_PCT))

q_high, q_medium

(0.867489410652155, 0.31064582137853425)

Assign bands to validation and test

In [11]:
def assign_band(score, q_high, q_medium):
    if score >= q_high:
        return "High"
    elif score >= q_medium:
        return "Medium"
    else:
        return "Low"

val_scored["risk_band"] = val_scored["risk_score"].apply(lambda s: assign_band(s, q_high, q_medium))
test_scored["risk_band"] = test_scored["risk_score"].apply(lambda s: assign_band(s, q_high, q_medium))

val_scored["risk_band"].value_counts(normalize=True), test_scored["risk_band"].value_counts(normalize=True)

(risk_band
 Low       0.799998
 Medium    0.149999
 High      0.050003
 Name: proportion, dtype: float64,
 risk_band
 Low       0.805671
 Medium    0.144007
 High      0.050322
 Name: proportion, dtype: float64)

Band performance summary

In [12]:
def band_summary(df):
    return (
        df.groupby("risk_band")
          .agg(
              n_customers=("risk_score", "size"),
              avg_score=("risk_score", "mean"),
              cease_rate=(TARGET, "mean")
          )
          .sort_values("avg_score", ascending=False)
    )

band_summary_val = band_summary(val_scored)
band_summary_test = band_summary(test_scored)

band_summary_val, band_summary_test

(           n_customers  avg_score  cease_rate
 risk_band                                    
 High             14850   0.933081    0.442222
 Medium           44547   0.585343    0.129436
 Low             237585   0.097078    0.007092,
            n_customers  avg_score  cease_rate
 risk_band                                    
 High             13171   0.933126    0.316529
 Medium           37692   0.583817    0.095219
 Low             210874   0.099111    0.005771)

Call list KPIs (High band only)

In [13]:
def call_list_kpis(df):
    called = df[df["risk_band"] == "High"]
    precision = float(called[TARGET].mean())  # hit rate among called
    recall = float(called[TARGET].sum() / df[TARGET].sum())  # share of all ceases captured
    return len(called), precision, recall

n_call_val, prec_val, rec_val = call_list_kpis(val_scored)
n_call_test, prec_test, rec_test = call_list_kpis(test_scored)

print("VAL  High-risk call list:", n_call_val, "precision:", round(prec_val, 3), "recall:", round(rec_val, 3))
print("TEST High-risk call list:", n_call_test, "precision:", round(prec_test, 3), "recall:", round(rec_test, 3))

VAL  High-risk call list: 14850 precision: 0.442 recall: 0.468
TEST High-risk call list: 13171 precision: 0.317 recall: 0.465


Add recommended intervention per band

In [14]:
ACTION_MAP = {
    "High":   "Outbound call within 48h + tailored retention offer/support",
    "Medium": "Targeted email/SMS + self-serve retention options; monitor for escalation",
    "Low":    "No proactive outreach; monitor monthly and include in general comms",
}

val_scored["recommended_action"] = val_scored["risk_band"].map(ACTION_MAP)
test_scored["recommended_action"] = test_scored["risk_band"].map(ACTION_MAP)

val_scored.head()

,unique_customer_identifier,snapshot_date,target_30d,risk_score,risk_band,recommended_action
0,2147b7e439b1e4dd597e88ee7cf15a4fc23d27046e7e17...,2024-05-01,0,0.083137,Low,No proactive outreach; monitor monthly and inc...
1,214fce5d267b45ba497710c529863fc1c9e733c3255beb...,2024-06-01,1,0.844723,Medium,Targeted email/SMS + self-serve retention opti...
2,21562ee19c61b797d76598c8be6fa2a492ceb840f52938...,2024-05-01,0,0.073508,Low,No proactive outreach; monitor monthly and inc...
3,21581257132d92a65c95b626d57dc452e717f6ed94e49f...,2024-06-01,0,0.273790,Low,No proactive outreach; monitor monthly and inc...
4,215b291bcfeb532e61e9444947bc23f55350f644f29efe...,2024-05-01,0,0.184566,Low,No proactive outreach; monitor monthly and inc...


Export an operational call list CSV (Top 5% highest risk)

In [15]:
REPORTS_DIR = PROJECT_ROOT / "reports"
REPORTS_DIR.mkdir(exist_ok=True)

call_list_val = (
    val_scored[val_scored["risk_band"] == "High"]
    .sort_values("risk_score", ascending=False)
    .reset_index(drop=True)
)

out_csv = REPORTS_DIR / "call_list_val_top5pct.csv"
call_list_val.to_csv(out_csv, index=False)

out_csv, call_list_val.head(10)

(WindowsPath('c:/Users/icons/Documents/UKTelecomCeasePrediction/reports/call_list_val_top5pct.csv'),
                           unique_customer_identifier snapshot_date  \
 0  a2f6025d7c7069cf1c2a040d2ceacd874ce16747c18409...    2024-05-01   
 1  7132b32dcf29676c5a074a96a5807b9126bebf2e0a49e9...    2024-05-01   
 2  93dfb7064b8f231522549f3721ddb650646a3663cdf85b...    2024-04-01   
 3  e8006542a2f987550387cd6379a71eff646ab877155312...    2024-06-01   
 4  8ff6a017385ed5057b4172c7624f3bbfa2f6552b0a1115...    2024-06-01   
 5  04f7948c0990be80bcdee3bf78f67189f9c77026b638f2...    2024-04-01   
 6  e32f94e1710938ad6b72d2ece85137900be7002a2c3388...    2024-06-01   
 7  732aff01f6164e7c686a7d975fa1a155071006058e9fbe...    2024-05-01   
 8  54b5cfdcf45a77d64c560c190aa8ad7c60e7aa630553b9...    2024-04-01   
 9  c7e02819167b31bfacbd95055fb6e766cb5d2143147923...    2024-05-01   
 
    target_30d  risk_score risk_band  \
 0           0    0.989289      High   
 1           1    0.988103      High

Save band thresholds into metadata for Streamlit consistency

In [16]:
meta["risk_band_policy"] = {"high_top_pct": HIGH_PCT, "medium_top_pct": MEDIUM_TOP_PCT}
meta["band_thresholds_from_val"] = {"q_high": q_high, "q_medium": q_medium}
meta["business_kpis_val"] = {"call_list_size": int(n_call_val), "precision": prec_val, "recall": rec_val}
meta["business_kpis_test"] = {"call_list_size": int(n_call_test), "precision": prec_test, "recall": rec_test}

with open(META_PATH, "w") as f:
    json.dump(meta, f, indent=2)

print("Updated metadata:", META_PATH)

Updated metadata: c:\Users\icons\Documents\UKTelecomCeasePrediction\models\metadata.json
